In [ ]:
q = 256
F.<a> = GF(q)

In [ ]:
def random_vector(n, F):
    """
    Retourne un vecteur tiré uniformément dans F^n.
    Entrées :
        n : longueur du vecteur
        F : le corps fini (ex: GF(256))
    """
    # On crée une liste de n éléments aléatoires du corps F
    v_list = [F.random_element() for _ in range(n)]
    
    # On convertit en objet vecteur de SageMath
    return vector(F, v_list)

# Test rapide

In [ ]:
def weight(x):
    """
    Calcule le poids de Hamming d'un mot x.
    """
    # On compte le nombre d'éléments qui ne sont pas égaux à l'élément nul du corps
    return len([c for c in x if c != 0])

# Test rapide
print(weight(v_test))

In [ ]:
import random

def random_fixed_weight(n, w, F):
    """
    Retourne un vecteur de F^n de poids exactement w.
    """
    # Initialisation avec un vecteur nul
    res = vector(F, [0]*n)
    
    # Choix de w positions distinctes parmi n
    positions = random.sample(range(n), int(w))
    
    for pos in positions:
        # On choisit une valeur non nulle
        val = F.random_element()
        while val == 0:
            val = F.random_element()
        res[pos] = val
        
    return res
test_vect = random_fixed_weight(10, 5, F)
print(test_vect)

In [ ]:
def random_matrix_max_rank(r, n, F):
    """
    Retourne une matrice aléatoire de taille r x n sur F de rang maximal r.
    """
    # On génère des matrices jusqu'à en trouver une de rang plein
    # Pour un corps assez grand (comme GF(256)), la probabilité est très élevée dès le premier essai
    while True:
        M = matrix(F, r, n, [F.random_element() for _ in range(r*n)])
        if M.rank() == r:
            return M

In [ ]:
import random

def random_subset(n, k):
    """
    Retourne un sous-ensemble aléatoire d'indices de cardinal k.
    """
    # On utilise random.sample pour garantir k indices distincts
    # On ajuste l'indexation (0 à n-1) pour Python/Sage
    return sorted(random.sample(range(n), k))

In [ ]:
def is_information_set(H, I):
    """
    Teste si I est un ensemble d'information pour le code de matrice de parité H.
    Cela revient à vérifier si la sous-matrice H_J est inversible.
    """
    n = H.ncols()
    # On définit J comme le complémentaire de I
    J = [j for j in range(n) if j not in I]
    
    # On extrait les colonnes correspondant à J
    H_J = H.matrix_from_columns(J)
    
    # H_J doit être carrée (n-k x n-k) et de rang plein
    return H_J.is_square() and H_J.rank() == H_J.nrows()

In [ ]:
import time
import random

def prange(H, w, s):
    """
    Implémente l'algorithme de Prange pour le décodage par syndrome.
    Entrées :
        H : Matrice de parité (r x n)
        w : Poids de l'erreur recherchée
        s : Syndrome cible (vecteur de taille r)
    Sortie :
        e : Vecteur d'erreur de poids w tel que H*e^T = s
        stats : Dictionnaire contenant le nombre d'itérations et le temps écoulé
    """
    r = H.nrows()
    n = H.ncols()
    k = n - r
    F = H.base_ring()
    
    nb_iterations = 0
    start_time = time.time()
    
    while True:
        nb_iterations += 1
        
        # 1. Choisir un ensemble d'information I de cardinal k [cite: 532, 534]
        I = random.sample(range(n), k)
        
        # 2. Vérifier si I est un ensemble d'information [cite: 533]
        # J est le complémentaire de I (les colonnes de parité)
        J = sorted([j for j in range(n) if j not in I])
        H_J = H.matrix_from_columns(J)
        
        if H_J.rank() == r:
            # 3. Résolution du système H_J * e_J = s 
            # On calcule e_J = H_J^-1 * s
            try:
                e_J = H_J.solve_right(s)
                
                # 4. Construire le vecteur d'erreur complet e 
                e = vector(F, n)
                for idx, val in enumerate(J):
                    e[val] = e_J[idx]
                
                # 5. Vérifier le poids de Hamming [cite: 529, 534]
                if e.hamming_weight() == w:
                    end_time = time.time()
                    execution_stats = {
                        "iterations": nb_iterations,
                        "time": end_time - start_time
                    }
                    return e, execution_stats
            except ValueError:
                # Cas où le système n'a pas de solution (rare si rang maximal)
                continue

# Exemple de test (Question 8) [cite: 535]
def test_prange():
    F = GF(2)
    n, k, w = 20, 10, 2
    r = n - k
    H = random_matrix_max_rank(r, n, F) # Ta fonction de la Q4
    e_reel = random_fixed_weight(n, w, F) # Ta fonction de la Q3
    s = H * e_reel
    
    e_trouve, stats = prange(H, w, s)
    print(f"Succès : {e_trouve == e_reel}")
    print(f"Nombre d'itérations : {stats['iterations']}")
    print(f"Temps : {stats['time']:.4f}s")
test_prange()

In [ ]:
import numpy as np

def benchmark_prange(q, n_list, k_func, w_func, iterations=50):
    """
    Exécute Prange plusieurs fois et retourne les médianes.
    """
    F = GF(q)
    resultats = []
    
    for n in n_list:
        k = k_func(n)
        w = w_func(n)
        r = n - k
        
        ite_list = []
        time_list = []
        
        for _ in range(iterations):
            # Génération des données de test
            H = random_matrix_max_rank(r, n, F)
            e_reel = random_fixed_weight(n, w, F)
            s = H * e_reel
            
            # Exécution
            _, stats = prange(H, w, s)
            ite_list.append(stats['iterations'])
            time_list.append(stats['time'])
            
        # Calcul des médianes 
        resultats.append({
            "n": n, "w": w,
            "med_ite": np.median(ite_list),
            "med_time": np.median(time_list)
        })
    return resultats

In [ ]:
print("--- Scénario (a) : n croissant ---")
n_vals = [20, 40, 60, 80] # À ajuster selon la puissance de calcul
res_a = benchmark_prange(2, n_vals, lambda n: n//2, lambda n: int(0.11*n))

for r in res_a:
    print(f"n={r['n']}, w={r['w']} | Médiane Itérations: {r['med_ite']}, Temps: {r['med_time']:.4f}s")

In [ ]:
print("\n--- Scénario (b) : w variant ---")
n_fixe = 60
w_vals = range(1, n_fixe//4 + 1, 2)
res_b = benchmark_prange(2, [n_fixe]*len(w_vals), lambda n: n//2, lambda n: w_vals[n_fixe.index(n) if isinstance(n_fixe, list) else 0])
# Note: w_vals doit être passé manuellement dans la boucle simplifiée ci-dessus